# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.16 — FAST
## General Spherically Symmetric Schwarzschild Radial-Vector Ansatz Audit

### Point de départ canonique

`.3.3.15` a **réfuté** le candidat statique aligné :

\[
u^\mu=n^\mu,\qquad u^r=0,
\]

car l'équation de lapse impose :

\[
c_1+c_4=0,
\]

alors que la branche faible champ stable exige :

\[
c_1+c_4>0.
\]

`.3.3.16` élargit donc l'ansatz à :

\[
\boxed{
u^\mu=(u^t(r),u^r(r),0,0),
\qquad u^r(r)\neq0\ \text{autorisé}.
}
\]

Le but de ce notebook est :

1. matérialiser exactement le système radial général ;
2. dériver les invariants GVH sans choisir encore un profil particulier ;
3. construire les équations réduites nécessaires sur fond Schwarzschild ;
4. vérifier que `.3.3.15` est bien récupéré comme sous-cas ;
5. tester un second candidat radial naturel : le champ de chute libre depuis l'infini ;
6. décider si le benchmark Schwarzschild est fermé ou si un problème ODE général reste ouvert.

Aucun PASS Schwarzschild n'est supposé à l'avance.

In [1]:
# RAD16.1 — Environment and canonical upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3315": {
        "canonical_user_executed_sha256": "6cb4e13d85dadf4e767570536591deb452743717946037353ff662522344a24e",
        "canonical_user_executed_size_bytes": 19956,
        "SCHWARZSCHILD_GR_GEOMETRY_VERIFIED": True,
        "SCHWARZSCHILD_BENCHMARK_PASS": False,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": False,
        "STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH": True,
        "KERR_BENCHMARK_AUTHORIZED": False,
        "NEXT_AUTHORIZED": "AUDIT-GENERAL-SPHERICALLY-SYMMETRIC-SCHWARZSCHILD-RADIAL-VECTOR-ANSATZ",
    }
}

UPSTREAM_GATE = all([
    UPSTREAM["p3315"]["SCHWARZSCHILD_GR_GEOMETRY_VERIFIED"],
    not UPSTREAM["p3315"]["SCHWARZSCHILD_BENCHMARK_PASS"],
    not UPSTREAM["p3315"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"],
    UPSTREAM["p3315"]["STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH"],
    not UPSTREAM["p3315"]["KERR_BENCHMARK_AUTHORIZED"],
])
assert UPSTREAM_GATE

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3315_CANONICAL_SHA256 =", UPSTREAM["p3315"]["canonical_user_executed_sha256"])

Python = 3.13.15
SymPy = 1.14.0
UPSTREAM_GATE = True
P3315_CANONICAL_SHA256 = 6cb4e13d85dadf4e767570536591deb452743717946037353ff662522344a24e


# RAD16.2 — Paramétrisation radiale par rapidité

Pour la métrique statique sphérique :

\[
ds^2=-N(r)^2dt^2+A(r)^2dr^2+r^2d\Omega^2,
\]

on paramètre le vecteur unitaire par une rapidité radiale \(\eta(r)\) :

\[
\boxed{
u^t=\frac{\cosh\eta}{N},
\qquad
u^r=\frac{\sinh\eta}{A}.
}
\]

Alors :

\[
-N^2(u^t)^2+A^2(u^r)^2
=
-\cosh^2\eta+\sinh^2\eta
=-1.
\]

La contrainte de norme est donc satisfaite identiquement.

Le sous-cas \(\eta=0\) reproduit exactement le vecteur aligné de `.3.3.15`.

In [2]:
# RAD16.3 — Exact unit-norm identity
eta,Nsym,Asym = sp.symbols("eta N A", real=True, nonzero=True)
ut = sp.cosh(eta)/Nsym
ur = sp.sinh(eta)/Asym
unit_norm = sp.simplify(-Nsym**2*ut**2 + Asym**2*ur**2)

RADIAL_UNIT_NORM_IDENTITY_PASS = (unit_norm == -1)
assert RADIAL_UNIT_NORM_IDENTITY_PASS

print("unit_norm =", unit_norm)
print("RADIAL_UNIT_NORM_IDENTITY_PASS =", RADIAL_UNIT_NORM_IDENTITY_PASS)

unit_norm = -1
RADIAL_UNIT_NORM_IDENTITY_PASS = True


# RAD16.4 — Invariants radiaux généraux

On définit :

\[
\nu(r)\equiv \frac{N'}{N},
\qquad
\eta'=\frac{d\eta}{dr}.
\]

Les invariants du secteur vectoriel se réduisent exactement à :

\[
\boxed{
I_1=
\frac1{A^2}
\left[
\eta'^2-\nu^2+\frac{2\sinh^2\eta}{r^2}
\right]
}
\]

\[
\boxed{
\theta=
\frac1A
\left[
\cosh\eta\,\eta'
+\sinh\eta\,\nu
+\frac{2\sinh\eta}{r}
\right]
}
\]

\[
\boxed{
I_3=
\frac1{A^2}
\left[
(\cosh\eta\,\eta'+\sinh\eta\,\nu)^2
+\frac{2\sinh^2\eta}{r^2}
\right]
}
\]

et

\[
\boxed{
a^2=
\frac1{A^2}
\left[
\sinh\eta\,\eta'
+\cosh\eta\,\nu
\right]^2.
}
\]

Ces formules contiennent le sous-cas aligné et tout profil radial statique unitaire.

In [3]:
# RAD16.5 — Symbolic invariant ledger
r,M = sp.symbols("r M", positive=True)
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
e,ep,nu,A = sp.symbols("e ep nu A", real=True, nonzero=True)

sh = sp.sinh(e)
ch = sp.cosh(e)

I1 = sp.factor((ep**2 - nu**2 + 2*sh**2/r**2)/A**2)
theta = sp.factor((ch*ep + sh*nu + 2*sh/r)/A)
I3 = sp.factor(((ch*ep + sh*nu)**2 + 2*sh**2/r**2)/A**2)
a2 = sp.factor((sh*ep + ch*nu)**2/A**2)

Lu = sp.factor(-c1*I1 - c2*theta**2 - c3*I3 + c4*a2)

RADIAL_INVARIANTS_MATERIALIZED = True
print("RADIAL_INVARIANTS_MATERIALIZED =", RADIAL_INVARIANTS_MATERIALIZED)
print("Lu operation count =", sp.count_ops(Lu))

RADIAL_INVARIANTS_MATERIALIZED = True
Lu operation count = 122


# RAD16.6 — Action radiale réduite et système nécessaire

Après intégration angulaire, à un facteur global \(4\pi\) près :

\[
L_{\rm rad}
=
NAr^2\,\mathcal L_u.
\]

Comme tous les invariants ci-dessus contiennent \(1/A^2\), on peut écrire :

\[
\boxed{
L_{\rm rad}
=
\frac{Nr^2}{A}\,F
}
\]

où :

\[
\begin{aligned}
F={}&
-c_1\left(
\eta'^2-\nu^2+\frac{2\sinh^2\eta}{r^2}
\right)\\
&-c_2\left(
\cosh\eta\,\eta'
+\sinh\eta\,\nu
+\frac{2\sinh\eta}{r}
\right)^2\\
&-c_3\left[
(\cosh\eta\,\eta'+\sinh\eta\,\nu)^2
+\frac{2\sinh^2\eta}{r^2}
\right]\\
&+c_4(
\sinh\eta\,\eta'
+\cosh\eta\,\nu
)^2.
\end{aligned}
\]

Sur une géométrie Schwarzschild déjà GR-vide, trois équations nécessaires du secteur vectoriel sont :

\[
\mathcal E_A=0,\qquad
\mathcal E_N=0,\qquad
\mathcal E_\eta=0.
\]

En particulier, puisque \(A\) n'apparaît ici que par le préfacteur \(1/A\),

\[
\boxed{
\mathcal E_A=0\Longrightarrow F=0.
}
\]

In [4]:
# RAD16.7 — Exact reduced functional and general Euler operators
X = sp.Function("X")(r)
E = sp.Function("E")(r)

f = sp.simplify(1-2*M/r)
A_S = 1/sp.sqrt(f)
N_S = sp.sqrt(f)

nu_fun = sp.diff(X,r)/X
ep_fun = sp.diff(E,r)
shf = sp.sinh(E)
chf = sp.cosh(E)

F_fun = (
    -c1*(ep_fun**2 - nu_fun**2 + 2*shf**2/r**2)
    -c2*(chf*ep_fun + shf*nu_fun + 2*shf/r)**2
    -c3*((chf*ep_fun + shf*nu_fun)**2 + 2*shf**2/r**2)
    +c4*(shf*ep_fun + chf*nu_fun)**2
)

Lrad_general = sp.expand(X*r**2/A_S * F_fun)

E_N_general = sp.diff(Lrad_general,X) - sp.diff(
    sp.diff(Lrad_general,sp.diff(X,r)),r
)
E_eta_general = sp.diff(Lrad_general,E) - sp.diff(
    sp.diff(Lrad_general,sp.diff(E,r)),r
)

GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED = True

print("GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED =", GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED)
print("ops(F) =", sp.count_ops(F_fun))
print("ops(E_N) =", sp.count_ops(E_N_general))
print("ops(E_eta) =", sp.count_ops(E_eta_general))

GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED = True
ops(F) = 78
ops(E_N) = 701
ops(E_eta) = 658


# RAD16.8 — Cross-check du sous-cas aligné

On impose :

\[
\eta=0,\qquad \eta'=0,
\qquad N=\sqrt f.
\]

Le notebook doit retrouver exactement le résidu de `.3.3.15` :

\[
\boxed{
\mathcal E_N
=
\frac{M^2(c_1+c_4)}
{\sqrt r\,(r-2M)^{3/2}}.
}
\]

In [5]:
# RAD16.9 — Aligned subcase reproduces .3.3.15
subs_aligned = {
    X:N_S,
    sp.diff(X,r):sp.diff(N_S,r),
    sp.diff(X,r,2):sp.diff(N_S,r,2),
    E:sp.Integer(0),
    sp.diff(E,r):sp.Integer(0),
    sp.diff(E,r,2):sp.Integer(0),
}

EN_aligned = sp.factor(sp.simplify(E_N_general.subs(subs_aligned).doit()))
expected_aligned = sp.factor(
    (c1+c4)*M**2/(sp.sqrt(r)*(r-2*M)**sp.Rational(3,2))
)

ALIGNED_SUBCASE_REPRODUCES_P3315 = (
    sp.simplify(EN_aligned-expected_aligned) == 0
)
assert ALIGNED_SUBCASE_REPRODUCES_P3315

print("E_N(aligned) =", EN_aligned)
print("ALIGNED_SUBCASE_REPRODUCES_P3315 =", ALIGNED_SUBCASE_REPRODUCES_P3315)

E_N(aligned) = M**2*(c1 + c4)/(sqrt(r)*(-2*M + r)**(3/2))
ALIGNED_SUBCASE_REPRODUCES_P3315 = True


# RAD16.10 — Candidat radial naturel : chute libre depuis l'infini

Le second profil exact testé est le champ géodésique radial avec énergie spécifique \(E=1\) :

\[
u_t=-1,
\]

donc :

\[
\boxed{
u^t=\frac1f,
\qquad
u^r=-\sqrt{\frac{2M}{r}}.
}
\]

Dans la paramétrisation par rapidité :

\[
\cosh\eta=\frac1{\sqrt f},
\]

\[
\sinh\eta=
-\sqrt{\frac{2M}{r-2M}}.
\]

Ce champ :

- est unitaire ;
- est radial ;
- est asymptotiquement aligné avec le temps de Schwarzschild ;
- possède \(u^r\neq0\) à rayon fini ;
- est géodésique, donc \(a^2=0\).

Il constitue donc un test radial beaucoup moins restrictif que `.3.3.15`.

In [6]:
# RAD16.11 — Exact free-fall invariants
y = -sp.sqrt(2*M/(r-2*M))
eta_ff = sp.asinh(y)

nu_S = sp.factor(sp.diff(sp.log(N_S),r))
ep_ff = sp.simplify(sp.diff(eta_ff,r))

subs_inv_ff = {
    e:eta_ff,
    ep:ep_ff,
    nu:nu_S,
    A:A_S,
}

I1_ff = sp.factor(sp.simplify(I1.subs(subs_inv_ff)))
theta_ff = sp.factor(sp.simplify(theta.subs(subs_inv_ff)))
I3_ff = sp.factor(sp.simplify(I3.subs(subs_inv_ff)))
a2_ff = sp.factor(sp.simplify(a2.subs(subs_inv_ff)))
Lu_ff = sp.factor(sp.simplify(Lu.subs(subs_inv_ff)))

c123 = sp.factor(c1+c2+c3)

FREEFALL_GEODESIC_A2_ZERO_PASS = (a2_ff == 0)
FREEFALL_INVARIANT_EQUALITY_PASS = (
    sp.simplify(I1_ff - I3_ff) == 0
    and sp.simplify(I1_ff - theta_ff**2) == 0
)
FREEFALL_LAGRANGIAN_CROSSCHECK_PASS = (
    sp.simplify(Lu_ff + sp.Rational(9,2)*M*c123/r**3) == 0
)

assert FREEFALL_GEODESIC_A2_ZERO_PASS
assert FREEFALL_INVARIANT_EQUALITY_PASS
assert FREEFALL_LAGRANGIAN_CROSSCHECK_PASS

print("I1_ff =", I1_ff)
print("theta_ff^2 =", sp.factor(theta_ff**2))
print("I3_ff =", I3_ff)
print("a2_ff =", a2_ff)
print("Lu_ff =", Lu_ff)

I1_ff = 9*M/(2*r**3)
theta_ff^2 = 9*M/(2*r**3)
I3_ff = 9*M/(2*r**3)
a2_ff = 0
Lu_ff = -9*M*(c1 + c2 + c3)/(2*r**3)


Le profil de chute libre donne :

\[
\boxed{
I_1=I_3=\theta^2=\frac{9M}{2r^3}
}
\]

et

\[
\boxed{
a^2=0.
}
\]

Ainsi :

\[
\boxed{
\mathcal L_u^{\rm ff}
=
-\frac{9M}{2r^3}(c_1+c_2+c_3).
}
\]

Il faut maintenant tester les équations nécessaires, pas seulement le lagrangien.

In [7]:
# RAD16.12 — Necessary equations on free-fall profile
subs_ff = {
    X:N_S,
    sp.diff(X,r):sp.diff(N_S,r),
    sp.diff(X,r,2):sp.diff(N_S,r,2),
    E:eta_ff,
    sp.diff(E,r):sp.diff(eta_ff,r),
    sp.diff(E,r,2):sp.diff(eta_ff,r,2),
}

# A-equation is F=0.
F_ff = sp.factor(sp.simplify(F_fun.subs(subs_ff).doit()))
EN_ff = sp.factor(sp.simplify(E_N_general.subs(subs_ff).doit()))
Eeta_ff = sp.factor(sp.simplify(E_eta_general.subs(subs_ff).doit()))

F_ff_expected = sp.factor(-sp.Rational(9,2)*M*c123/(r**2*(r-2*M)))
EN_ff_expected = sp.factor(-sp.Rational(9,2)*M*c123/(sp.sqrt(r)*sp.sqrt(r-2*M)))
Eeta_ff_expected = sp.factor(sp.Rational(9,2)*sp.sqrt(2)*sp.sqrt(M)*c123/sp.sqrt(r))

FREEFALL_A_EOM_CROSSCHECK_PASS = sp.simplify(F_ff-F_ff_expected)==0
FREEFALL_LAPSE_EOM_CROSSCHECK_PASS = sp.simplify(EN_ff-EN_ff_expected)==0
FREEFALL_VECTOR_EOM_CROSSCHECK_PASS = sp.simplify(Eeta_ff-Eeta_ff_expected)==0

FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO = all([
    FREEFALL_A_EOM_CROSSCHECK_PASS,
    FREEFALL_LAPSE_EOM_CROSSCHECK_PASS,
    FREEFALL_VECTOR_EOM_CROSSCHECK_PASS,
])

assert FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO

print("F_ff =", F_ff)
print("E_N_ff =", EN_ff)
print("E_eta_ff =", Eeta_ff)
print("FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO =", FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO)

F_ff = -9*M*(c1 + c2 + c3)/(2*r**2*(-2*M + r))
E_N_ff = -9*M*(c1 + c2 + c3)/(2*sqrt(r)*sqrt(-2*M + r))
E_eta_ff = 9*sqrt(2)*sqrt(M)*(c1 + c2 + c3)/(2*sqrt(r))
FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO = True


# RAD16.13 — Confrontation avec la branche faible champ générique

La fermeture scalaire de `.3.3.14` utilise :

\[
K_S=
\frac{
2(1-c_1-c_3)(c_1+3c_2+c_3+2)
}{
c_1+c_2+c_3
}.
\]

La branche générique faible champ exige donc :

\[
\boxed{
c_1+c_2+c_3\neq0
}
\]

afin que le mode scalaire réduit soit non singulier.

Mais le candidat radial géodésique de chute libre impose :

\[
\boxed{
c_1+c_2+c_3=0.
}
\]

Donc ce **second candidat analytique** est lui aussi incompatible avec la branche faible champ générique stable.

In [8]:
# RAD16.14 — Free-fall branch compatibility classifier
WEAK_FIELD_GENERIC_SCALAR_REQUIRES_C123_NONZERO = True
FREEFALL_SCHWARZSCHILD_REQUIRES_C123_ZERO = FREEFALL_NECESSARY_EOMS_REQUIRE_C123_ZERO

FREEFALL_COMPATIBLE_WITH_GENERIC_STABLE_WEAK_FIELD_BRANCH = False

FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH = all([
    WEAK_FIELD_GENERIC_SCALAR_REQUIRES_C123_NONZERO,
    FREEFALL_SCHWARZSCHILD_REQUIRES_C123_ZERO,
    not FREEFALL_COMPATIBLE_WITH_GENERIC_STABLE_WEAK_FIELD_BRANCH,
])

assert FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH

print("weak-field generic requirement: c1+c2+c3 != 0")
print("free-fall Schwarzschild necessary condition: c1+c2+c3 = 0")
print("FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH =", FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH)

weak-field generic requirement: c1+c2+c3 != 0
free-fall Schwarzschild necessary condition: c1+c2+c3 = 0
FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH = True


# RAD16.15 — Ce qui est maintenant fermé et ce qui reste ouvert

Deux sous-familles exactes sont désormais exclues sur la branche faible champ stable/générique :

### 1. Alignée statique
\[
\eta=0
\quad\Rightarrow\quad
c_1+c_4=0,
\]
incompatible avec :
\[
c_1+c_4>0.
\]

### 2. Radiale géodésique, chute libre depuis l'infini
\[
u_t=-1
\quad\Rightarrow\quad
c_1+c_2+c_3=0,
\]
incompatible avec la branche scalaire générique :
\[
c_1+c_2+c_3\neq0.
\]

Mais ces deux résultats **ne constituent pas encore un théorème no-go pour toute fonction \(\eta(r)\)**.

Le système général :

\[
F[\eta,\eta';r]=0,
\]

\[
\mathcal E_N[\eta,\eta',\eta'';r]=0,
\]

\[
\mathcal E_\eta[\eta,\eta',\eta'';r]=0
\]

reste un problème d'existence avec conditions aux limites.

In [9]:
# RAD16.16 — Final classifier
GENERAL_RADIAL_ANSATZ_MATERIALIZED = all([
    RADIAL_UNIT_NORM_IDENTITY_PASS,
    RADIAL_INVARIANTS_MATERIALIZED,
    GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED,
    ALIGNED_SUBCASE_REPRODUCES_P3315,
])

TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED = all([
    UPSTREAM["p3315"]["STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH"],
    FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH,
])

GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED = False
FULL_SCHWARZSCHILD_GVH_FIELD_EQUATIONS_CLOSED = False

SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False
SCHWARZSCHILD_BENCHMARK_STATUS = (
    "GENERAL_RADIAL_SYSTEM_MATERIALIZED_ALIGNED_AND_FREEFALL_SUBBRANCHES_REFUTED_GENERAL_ODE_EXISTENCE_PENDING"
)

KERR_BENCHMARK_AUTHORIZED = False
CLASSICAL_PREDICTIONS_AUTHORIZED = False
QUANTIZATION_READY = False

RAD16_OBSTRUCTIONS = [
    "GENERAL-RADIAL-ODE-EXISTENCE-NOT-YET-CLASSIFIED",
    "ASYMPTOTIC-BOUNDARY-CONDITIONS-NOT-YET-SOLVED",
    "HORIZON-REGULARITY-NOT-YET-CLASSIFIED",
    "FULL-SCHWARZSCHILD-GVH-FIELD-EQUATIONS-NOT-YET-CLOSED",
]

RAD16_LOCAL_AUDIT_PASS = all([
    UPSTREAM_GATE,
    GENERAL_RADIAL_ANSATZ_MATERIALIZED,
    TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED,
    not GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED,
    not SCHWARZSCHILD_BENCHMARK_PASS,
    not SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
    not KERR_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not QUANTIZATION_READY,
])

RAD16_NEXT_AUTHORIZED = (
    "AUDIT-SCHWARZSCHILD-GENERAL-RADIAL-ODE-EXISTENCE-BOUNDARY-AND-HORIZON-REGULARITY"
    if RAD16_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.16-GENERAL-RADIAL-ANSATZ-AUDIT"
)

assert RAD16_LOCAL_AUDIT_PASS

print("GENERAL_RADIAL_ANSATZ_MATERIALIZED =", GENERAL_RADIAL_ANSATZ_MATERIALIZED)
print("TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED =", TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED)
print("GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED =", GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED)
print("SCHWARZSCHILD_BENCHMARK_PASS =", SCHWARZSCHILD_BENCHMARK_PASS)
print("SCHWARZSCHILD_BENCHMARK_AUTHORIZED =", SCHWARZSCHILD_BENCHMARK_AUTHORIZED)
print("SCHWARZSCHILD_BENCHMARK_STATUS =", SCHWARZSCHILD_BENCHMARK_STATUS)
print("KERR_BENCHMARK_AUTHORIZED =", KERR_BENCHMARK_AUTHORIZED)
print("RAD16_OBSTRUCTIONS =", RAD16_OBSTRUCTIONS)
print("RAD16_NEXT_AUTHORIZED =", RAD16_NEXT_AUTHORIZED)

GENERAL_RADIAL_ANSATZ_MATERIALIZED = True
TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED = True
GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED = False
SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False
SCHWARZSCHILD_BENCHMARK_STATUS = GENERAL_RADIAL_SYSTEM_MATERIALIZED_ALIGNED_AND_FREEFALL_SUBBRANCHES_REFUTED_GENERAL_ODE_EXISTENCE_PENDING
KERR_BENCHMARK_AUTHORIZED = False
RAD16_OBSTRUCTIONS = ['GENERAL-RADIAL-ODE-EXISTENCE-NOT-YET-CLASSIFIED', 'ASYMPTOTIC-BOUNDARY-CONDITIONS-NOT-YET-SOLVED', 'HORIZON-REGULARITY-NOT-YET-CLASSIFIED', 'FULL-SCHWARZSCHILD-GVH-FIELD-EQUATIONS-NOT-YET-CLOSED']
RAD16_NEXT_AUTHORIZED = AUDIT-SCHWARZSCHILD-GENERAL-RADIAL-ODE-EXISTENCE-BOUNDARY-AND-HORIZON-REGULARITY


# RAD16.17 — Discipline scientifique

Un `RAD16_LOCAL_AUDIT_PASS=True` signifie uniquement que :

- le système radial général est matérialisé ;
- le sous-cas `.3.3.15` est reproduit ;
- un second candidat radial naturel a été testé et réfuté.

Cela **ne signifie pas** :

\[
\texttt{SCHWARZSCHILD\_BENCHMARK\_PASS=True}.
\]

Au contraire, Schwarzschild reste explicitement non validé tant que l'existence ou la non-existence d'une solution générale \(\eta(r)\) compatible avec :

\[
\eta(r)\to0\quad (r\to\infty)
\]

et une régularité appropriée à l'horizon n'est pas démontrée.

In [10]:
# RAD16.18 — Artifact JSON
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.16_General_Spherically_Symmetric_Schwarzschild_Radial_Vector_Ansatz_Audit_FAST",
    "execution_scope": "GENERAL_SPHERICALLY_SYMMETRIC_SCHWARZSCHILD_RADIAL_VECTOR_ANSATZ",
    "upstream": UPSTREAM,
    "general_radial_ansatz": {
        "parameterization": "u^t=cosh(eta)/N, u^r=sinh(eta)/A",
        "unit_norm_identity_pass": RADIAL_UNIT_NORM_IDENTITY_PASS,
        "invariants_materialized": RADIAL_INVARIANTS_MATERIALIZED,
        "reduced_system_materialized": GENERAL_RADIAL_REDUCED_SYSTEM_MATERIALIZED,
        "A_equation_necessary_condition": "F=0",
    },
    "aligned_subbranch": {
        "reproduces_p3315": ALIGNED_SUBCASE_REPRODUCES_P3315,
        "necessary_condition": "c1+c4=0",
        "refuted_on_stable_branch": True,
    },
    "freefall_subbranch": {
        "u_t": "-1",
        "u_t_contravariant": "u^t=1/f",
        "u_r_contravariant": "u^r=-sqrt(2M/r)",
        "I1": str(I1_ff),
        "theta_squared": str(sp.factor(theta_ff**2)),
        "I3": str(I3_ff),
        "a_squared": str(a2_ff),
        "L_u": str(Lu_ff),
        "F": str(F_ff),
        "E_N": str(EN_ff),
        "E_eta": str(Eeta_ff),
        "necessary_condition": "c1+c2+c3=0",
        "refuted_on_generic_stable_branch": FREEFALL_RADIAL_CANDIDATE_REFUTED_ON_GENERIC_STABLE_BRANCH,
    },
    "scientific_status": {
        "GENERAL_RADIAL_ANSATZ_MATERIALIZED": GENERAL_RADIAL_ANSATZ_MATERIALIZED,
        "TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED": TWO_NATURAL_RADIAL_SUBBRANCHES_REFUTED,
        "GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED": GENERAL_RADIAL_ODE_EXISTENCE_CLASSIFIED,
        "FULL_SCHWARZSCHILD_GVH_FIELD_EQUATIONS_CLOSED": FULL_SCHWARZSCHILD_GVH_FIELD_EQUATIONS_CLOSED,
        "SCHWARZSCHILD_BENCHMARK_PASS": SCHWARZSCHILD_BENCHMARK_PASS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "KERR_BENCHMARK_AUTHORIZED": KERR_BENCHMARK_AUTHORIZED,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": CLASSICAL_PREDICTIONS_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "RAD16_LOCAL_AUDIT_PASS": RAD16_LOCAL_AUDIT_PASS,
        "obstructions": RAD16_OBSTRUCTIONS,
    },
    "next_authorized": RAD16_NEXT_AUTHORIZED,
    "scope_note": "General radial system materialized. Static-aligned and asymptotically aligned geodesic free-fall subbranches are refuted on the inherited weak-field branch. No general radial no-go theorem yet."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.16_General_Spherically_Symmetric_Schwarzschild_Radial_Vector_Ansatz_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("RAD16 artifact =", artifact_path)

RAD16 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.16_General_Spherically_Symmetric_Schwarzschild_Radial_Vector_Ansatz_Audit_FAST.json


# Conclusion

`.3.3.16` ne promeut pas Schwarzschild.

Il établit :

\[
\boxed{
\text{ansatz radial général matérialisé}
}
\]

et deux exclusions exactes :

\[
\boxed{
\eta=0
\Rightarrow
c_1+c_4=0
}
\]

\[
\boxed{
u_t=-1
\Rightarrow
c_1+c_2+c_3=0.
}
\]

La prochaine étape admissible doit maintenant résoudre ou exclure le **système ODE radial général** avec conditions asymptotiques et régularité d'horizon.